# Laboratorio 4 — Cuaderno 6: análisis exploratorio adicional

**Ejercicio 8.** Extensión de la floración, zonas persistentes, comparación de distribuciones
entre fechas y búsqueda de patrón estacional.

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from src import config, datos, graficos
from src import indices as ix

resumen = datos.tabla_resumen()
pixeles = datos.tabla_pixeles()
COLOR = {"Atitlan": "#1f6f8b", "Amatitlan": "#c1272d"}
print(f"{len(resumen)} escenas | {len(pixeles):,} píxeles muestreados")

## 8.1 Extensión espacial de la floración

Una floración de 100 µg/L concentrada en una esquina no es lo mismo que una de 60 µg/L
repartida por todo el lago. Medimos qué **porcentaje del espejo de agua** supera cada nivel de
severidad en cada fecha.

In [ ]:
CORTES = [7, 20, 50, 100]

filas = []
for lago, fecha in datos.escenas_disponibles():
    capas, _ = datos.indices_escena(lago, fecha)
    valores = capas["chl_agua"][capas["agua_valida"]]
    if valores.size == 0:
        continue
    fila = {"lago": lago, "fecha": pd.Timestamp(fecha),
            "area_km2": valores.size * datos.AREA_PIXEL_KM2}
    for corte in CORTES:
        fila[f"pct_sobre_{corte}"] = 100 * np.mean(valores > corte)
        fila[f"km2_sobre_{corte}"] = np.sum(valores > corte) * datos.AREA_PIXEL_KM2
    filas.append(fila)

extension = pd.DataFrame(filas).sort_values(["lago", "fecha"])

for lago in ["Atitlan", "Amatitlan"]:
    print(f"\n{config.NOMBRE_LARGO[lago]}")
    sub = extension[extension["lago"] == lago].copy()
    sub["fecha"] = sub["fecha"].dt.strftime("%Y-%m-%d")
    print(sub.drop(columns="lago").round(2).to_string(index=False))

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(13, 9), sharex=False)

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = extension[extension["lago"] == lago]
    fondo = ["#c6dbef", "#6baed6", "#e8a33d", "#c1272d"]
    for corte, color in zip(CORTES, fondo):
        eje.plot(sub["fecha"], sub[f"pct_sobre_{corte}"], "o-", color=color,
                 linewidth=2, markersize=6, label=f"> {corte} µg/L")
    eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=12)
    eje.set_ylabel("% del espejo de agua")
    eje.legend(fontsize=9, ncol=4)
    eje.grid(alpha=0.3)
    eje.tick_params(axis="x", rotation=30)

fig.suptitle("Extensión de la floración por nivel de severidad", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Área absoluta afectada, que es lo que importa para gestión.
fig, eje = plt.subplots(figsize=(12, 5))
ancho = 12
for lago in ["Atitlan", "Amatitlan"]:
    sub = extension[extension["lago"] == lago]
    eje.bar(sub["fecha"], sub["km2_sobre_50"], width=ancho,
            color=COLOR[lago], alpha=0.85, label=config.NOMBRE_LARGO[lago])
eje.set_ylabel(f"km² sobre {ix.UMBRAL_ALTO_CHL:.0f} µg/L")
eje.set_title("Superficie en estado de alerta, en kilómetros cuadrados", loc="left")
eje.legend()
eje.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

## 8.2 Zonas persistentes de acumulación

En el cuaderno 4 mapeamos la persistencia. Aquí la cuantificamos: cuánta superficie del lago
está crónicamente afectada, frente a cuánta se afecta solo de forma ocasional.

In [ ]:
def perfil_persistencia(lago):
    pila, fechas, _ = datos.pila_chl(lago)
    observaciones = np.sum(np.isfinite(pila), axis=0)
    confiable = observaciones >= max(2, len(fechas) // 2)
    excesos = np.sum(pila > ix.UMBRAL_ALTO_CHL, axis=0)
    persistencia = np.where(confiable, 100 * excesos / np.maximum(observaciones, 1), np.nan)

    validos = persistencia[np.isfinite(persistencia)]
    area_total = validos.size * datos.AREA_PIXEL_KM2

    print(f"\n{config.NOMBRE_LARGO[lago]} — superficie analizada {area_total:.1f} km²")
    categorias = [
        ("Nunca supera el umbral", validos == 0),
        ("Ocasional (1-25% de fechas)", (validos > 0) & (validos <= 25)),
        ("Recurrente (25-50%)", (validos > 25) & (validos <= 50)),
        ("Frecuente (50-75%)", (validos > 50) & (validos <= 75)),
        ("Crónica (>75%)", validos > 75),
    ]
    filas = []
    for nombre, mascara in categorias:
        n = int(mascara.sum())
        filas.append({"categoría": nombre, "km²": round(n * datos.AREA_PIXEL_KM2, 2),
                      "% del lago": round(100 * n / validos.size, 1)})
    tabla = pd.DataFrame(filas)
    print(tabla.to_string(index=False))
    return persistencia, tabla

persistencia = {}
tablas = {}
for lago in ["Atitlan", "Amatitlan"]:
    persistencia[lago], tablas[lago] = perfil_persistencia(lago)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    tabla = tablas[lago]
    colores = ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"]
    eje.barh(tabla["categoría"], tabla["% del lago"], color=colores)
    for i, (_, f) in enumerate(tabla.iterrows()):
        if f["% del lago"] > 0:
            eje.text(f["% del lago"] + 1, i, f"{f['% del lago']:.1f}%  ({f['km²']:.1f} km²)",
                     va="center", fontsize=8)
    eje.set_xlabel("% de la superficie del lago")
    eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=12)
    eje.set_xlim(0, 118)
    eje.grid(alpha=0.3, axis="x")
fig.suptitle("Qué tan crónica es la afectación", fontsize=14)
fig.tight_layout()
plt.show()

## 8.3 Distribución de valores entre fechas

### Histogramas superpuestos

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 5))

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = pixeles[pixeles["lago"] == lago]
    fechas = sorted(sub["fecha"].unique())
    paleta = plt.cm.viridis(np.linspace(0, 1, len(fechas)))
    for fecha, color in zip(fechas, paleta):
        valores = sub[sub["fecha"] == fecha]["chl"].dropna()
        valores = valores[valores > 0]
        if len(valores) < 50:
            continue
        eje.hist(np.log10(valores), bins=70, histtype="step", density=True,
                 color=color, linewidth=1.6,
                 label=pd.Timestamp(fecha).strftime("%Y-%m-%d"))
    eje.axvline(np.log10(ix.UMBRAL_ALTO_CHL), color="#d95f02",
                linestyle="--", linewidth=1.5)
    eje.set_xlabel("log₁₀ de clorofila-a (µg/L)")
    eje.set_ylabel("Densidad")
    eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=12)
    eje.legend(fontsize=6.5, ncol=2)
    eje.grid(alpha=0.3)

fig.suptitle("Distribución de la clorofila en cada fecha", fontsize=14)
fig.tight_layout()
plt.show()

### Diagramas de caja por fecha

Los histogramas se saturan con 11 curvas. Los diagramas de caja dejan ver el desplazamiento de
la distribución completa fecha a fecha.

In [ ]:
fig, ejes = plt.subplots(2, 1, figsize=(13, 10))

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = pixeles[pixeles["lago"] == lago]
    fechas = sorted(sub["fecha"].unique())
    grupos = [sub[sub["fecha"] == f]["chl"].dropna().values for f in fechas]
    etiquetas = [pd.Timestamp(f).strftime("%d %b\n%y") for f in fechas]

    partes = eje.boxplot(grupos, tick_labels=etiquetas, patch_artist=True,
                         showfliers=False, widths=0.6)
    for caja in partes["boxes"]:
        caja.set_facecolor(COLOR[lago]); caja.set_alpha(0.6)

    eje.axhline(ix.UMBRAL_ALTO_CHL, color="#d95f02", linestyle="--",
                linewidth=1.5, label=f"Umbral {ix.UMBRAL_ALTO_CHL:.0f} µg/L")
    eje.set_ylabel("Clorofila-a (µg/L)")
    eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=12)
    eje.legend(fontsize=9)
    eje.grid(alpha=0.3, axis="y")
    eje.tick_params(axis="x", labelsize=8)

fig.suptitle("Distribución de la clorofila por fecha (sin valores atípicos)", fontsize=14)
fig.tight_layout()
plt.show()

## 8.4 ¿Hay patrón estacional?

Con 11 fechas por lago repartidas en 18 meses no se puede ajustar un modelo estacional formal.
Lo que sí se puede hacer es contrastar los dos regímenes climáticos de Guatemala y ver si la
diferencia es lo bastante grande como para no ser ruido.

In [ ]:
def estacion(fecha):
    return "Seca" if fecha.month in (11, 12, 1, 2, 3, 4) else "Lluviosa"

resumen["estacion"] = resumen["fecha"].apply(estacion)

for lago in ["Atitlan", "Amatitlan"]:
    sub = resumen[resumen["lago"] == lago]
    seca = sub[sub["estacion"] == "Seca"]["chl_media"].dropna()
    lluvia = sub[sub["estacion"] == "Lluviosa"]["chl_media"].dropna()

    print(f"\n{'='*58}\n{config.NOMBRE_LARGO[lago]}")
    print(f"  Seca     : n={len(seca):2d}  media {seca.mean():7.2f} µg/L  "
          f"rango {seca.min():.1f}–{seca.max():.1f}")
    print(f"  Lluviosa : n={len(lluvia):2d}  media {lluvia.mean():7.2f} µg/L  "
          f"rango {lluvia.min():.1f}–{lluvia.max():.1f}")

    if len(seca) >= 3 and len(lluvia) >= 3:
        u, p = stats.mannwhitneyu(seca, lluvia, alternative="two-sided")
        print(f"  Mann-Whitney U: p = {p:.3f}  →  "
              f"{'diferencia significativa al 5%' if p < 0.05 else 'no se puede afirmar diferencia'}")
    else:
        print("  Muy pocas observaciones en algún grupo para una prueba formal.")

In [ ]:
# Serie ordenada por día del año, para ver si hay una forma anual repetida.
fig, ejes = plt.subplots(1, 2, figsize=(15, 5))

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = resumen[resumen["lago"] == lago].copy()
    sub["dia_anio"] = sub["fecha"].dt.dayofyear
    for anio, marca in [(2025, "o"), (2026, "s")]:
        s = sub[sub["fecha"].dt.year == anio].sort_values("dia_anio")
        if len(s):
            eje.plot(s["dia_anio"], s["chl_media"], marca + "-", markersize=9,
                     linewidth=1.8, label=str(anio), alpha=0.85)
    eje.axvspan(121, 304, color="#4a7c9e", alpha=0.10)
    eje.text(212, eje.get_ylim()[1] * 0.96, "estación lluviosa",
             ha="center", fontsize=9, color="#33637f")
    eje.set_xlabel("Día del año")
    eje.set_ylabel("Clorofila-a media (µg/L)")
    eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=12)
    eje.set_xlim(0, 366)
    eje.legend(title="Año", fontsize=9)
    eje.grid(alpha=0.3)

fig.suptitle("Los dos años superpuestos sobre el calendario", fontsize=14)
fig.tight_layout()
plt.show()

### Advertencia metodológica

Hay un sesgo que no se puede eliminar y hay que declararlo: **las fechas disponibles no son
aleatorias**. Son las fechas en que el satélite pasó y encontró cielo despejado. En Guatemala el
cielo despejado es mucho más común en la estación seca, así que la estación lluviosa está
sistemáticamente submuestreada y, además, las pocas escenas de esa estación son las de días
atípicamente despejados.

Esto significa que cualquier conclusión estacional de este trabajo es **indicativa, no
concluyente**. Para afirmar un patrón estacional harían falta varios años completos de
observaciones y, de preferencia, datos de campo que no dependan de que no haya nubes.

## 8.5 Relación entre extensión e intensidad

Una última pregunta útil para gestión: cuando el promedio del lago sube, ¿es porque toda la
superficie sube un poco, o porque un foco pequeño se dispara?

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

for eje, lago in zip(ejes, ["Atitlan", "Amatitlan"]):
    sub = resumen[resumen["lago"] == lago].dropna(subset=["chl_media", "pct_alto"])
    eje.scatter(sub["chl_media"], sub["pct_alto"], s=100, color=COLOR[lago], alpha=0.85)
    for _, f in sub.iterrows():
        eje.annotate(f"{f['fecha']:%b %y}", (f["chl_media"], f["pct_alto"]),
                     textcoords="offset points", xytext=(7, 4), fontsize=7)
    if len(sub) > 2 and sub["pct_alto"].std() > 0:
        r, p = stats.spearmanr(sub["chl_media"], sub["pct_alto"])
        eje.set_title(f"{config.NOMBRE_LARGO[lago]}  (ρ = {r:+.2f}, p = {p:.3f})",
                      loc="left", fontsize=11)
    else:
        eje.set_title(config.NOMBRE_LARGO[lago], loc="left", fontsize=11)
    eje.set_xlabel("Clorofila-a media del lago (µg/L)")
    eje.set_ylabel(f"% del lago sobre {ix.UMBRAL_ALTO_CHL:.0f} µg/L")
    eje.grid(alpha=0.3)

fig.suptitle("¿La media sube porque sube todo, o por focos concentrados?", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Concentración de la carga: qué fracción de la clorofila total del lago está
# contenida en el 10% de píxeles más cargados.
filas = []
for lago, fecha in datos.escenas_disponibles():
    capas, _ = datos.indices_escena(lago, fecha)
    valores = capas["chl_agua"][capas["agua_valida"]]
    if valores.size < 100:
        continue
    ordenados = np.sort(valores)[::-1]
    n10 = max(1, len(ordenados) // 10)
    filas.append({
        "lago": lago,
        "fecha": pd.Timestamp(fecha),
        "pct_carga_en_10pct_area": 100 * ordenados[:n10].sum() / ordenados.sum(),
    })

concentracion = pd.DataFrame(filas)
promedios = concentracion.groupby("lago")["pct_carga_en_10pct_area"].mean().round(1)
print("Porcentaje de la clorofila total contenida en el 10% de superficie más cargada:\n")
for lago, valor in promedios.items():
    print(f"  {config.NOMBRE_LARGO[lago]:22s} {valor:5.1f}%")
print("\n(Si fuera 10%, la carga estaría repartida de forma perfectamente uniforme.)")
concentracion.pivot_table(index="fecha", columns="lago",
                          values="pct_carga_en_10pct_area").round(1)